# SSL preprocessing

**In plain language:** SSL means semi-supervised learning — extra unlabeled scans were labeled by a teacher, then a nnU-Net was trained on labeled CTs plus those teacher masks (Dice 0.471). nnU-Net Ensemble is separate: two folds averaged (locked-test Dice 0.489).

Research software — not for clinical use. Uses `data/demo/` only.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import yaml

cwd = Path.cwd().resolve()
if (cwd / "config" / "preprocessing.yaml").is_file():
    ROOT = cwd
elif (cwd.parent / "config" / "preprocessing.yaml").is_file():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the repo root or notebooks/")

CT = ROOT / "data" / "demo" / "01_all_classes" / "all_classes.nii.gz"
GT = ROOT / "data" / "demo" / "ground_truth" / "01_all_classes" / "all_classes.nii.gz"
cfg = yaml.safe_load((ROOT / "config" / "preprocessing.yaml").read_text(encoding="utf-8"))
volume = nib.load(CT).get_fdata().astype(np.float32)
mask = nib.load(GT).get_fdata().astype(np.int16)
windowed = np.clip(volume, cfg["clip_min"], cfg["clip_max"])
z = int(np.argmax((mask > 0).sum(axis=(0, 1))))
print("Same locked HU window as the labeled models:", cfg["clip_min"], "to", cfg["clip_max"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.rot90(windowed[:, :, z]), cmap="gray")
axes[0].set_title("Windowed CT")
axes[0].axis("off")
axes[1].imshow(np.rot90(mask[:, :, z]), cmap="nipy_spectral", vmin=0, vmax=5)
axes[1].set_title("Expert labels")
axes[1].axis("off")
plt.tight_layout()
plt.show()

**SSL** stands for semi-supervised learning. The **nnU-Net Ensemble** button averages two nnU-Net folds (locked-test Dice 0.489). **SSL** trains one nnU-Net on labeled CTs plus 800 teacher masks (Dice 0.471). IoU and volume error were not scored for SSL.